In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../datasets/dataset1/diabetic_data.csv")

df.replace("?", np.nan, inplace=True)

data = df.copy()

print("Dataset loaded successfully")
print("Shape:", data.shape)

Dataset loaded successfully
Shape: (101766, 50)


In [2]:
data.drop(["encounter_id", "patient_nbr"], axis=1, inplace=True)

print("Remaining columns:", data.shape[1])

Remaining columns: 48


In [3]:
constant_columns = [
    col for col in data.columns
    if data[col].nunique(dropna=False) <= 1
]

print("Constant columns:", constant_columns)

Constant columns: ['examide', 'citoglipton']


In [4]:
data.drop(columns=constant_columns, inplace=True)

print("Shape after removing constant columns:", data.shape)

Shape after removing constant columns: (101766, 46)


In [5]:
data["readmitted"] = data["readmitted"].map({
    "<30": 1,
    ">30": 0,
    "NO": 0
})

print(data["readmitted"].value_counts())

readmitted
0    90409
1    11357
Name: count, dtype: int64


In [6]:
print(data["readmitted"].value_counts(normalize=True) * 100)

readmitted
0    88.840084
1    11.159916
Name: proportion, dtype: float64


In [7]:
missing = data.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

weight               98569
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64


In [8]:
data.to_csv(
    "../datasets/dataset1/preprocessed_initial.csv",
    index=False
)

print("Initial preprocessing completed and saved!")

Initial preprocessing completed and saved!


In [10]:
data.drop(
    columns=["examide", "citoglipton"],
    inplace=True,
    errors="ignore"
)

print("Constant columns removed successfully!")
print("Dataset shape:", data.shape)

Constant columns removed successfully!
Dataset shape: (101766, 46)


In [11]:
print(data.columns.tolist())

['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [12]:
data.drop(columns=["weight"], inplace=True, errors="ignore")

print("Weight column removed.")
print("Dataset shape:", data.shape)

Weight column removed.
Dataset shape: (101766, 45)


In [13]:
# Replace missing categorical values

columns_to_fill = [
    "medical_specialty",
    "payer_code",
    "race",
    "diag_1",
    "diag_2",
    "diag_3"
]

for col in columns_to_fill:
    data[col] = data[col].fillna("Unknown")


# Handle missing laboratory test results

data["max_glu_serum"] = data["max_glu_serum"].fillna("Not Tested")

data["A1Cresult"] = data["A1Cresult"].fillna("Not Tested")

In [14]:
print("Remaining missing values:")

print(data.isnull().sum()[data.isnull().sum() > 0])

Remaining missing values:
Series([], dtype: int64)


In [15]:
print("Dataset Shape:", data.shape)

print("\nDataset Information:")
data.info()

Dataset Shape: (101766, 45)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 45 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   race                      101766 non-null  object
 1   gender                    101766 non-null  object
 2   age                       101766 non-null  object
 3   admission_type_id         101766 non-null  int64 
 4   discharge_disposition_id  101766 non-null  int64 
 5   admission_source_id       101766 non-null  int64 
 6   time_in_hospital          101766 non-null  int64 
 7   payer_code                101766 non-null  object
 8   medical_specialty         101766 non-null  object
 9   num_lab_procedures        101766 non-null  int64 
 10  num_procedures            101766 non-null  int64 
 11  num_medications           101766 non-null  int64 
 12  number_outpatient         101766 non-null  int64 
 13  number_em

In [16]:
data.to_csv(
    "../datasets/dataset1/preprocessed_initial.csv",
    index=False
)

print("Preprocessed dataset saved successfully!")

Preprocessed dataset saved successfully!


In [17]:
# Preserve patient IDs from the original dataset
patient_ids = df["patient_nbr"].copy()

# Separate input features and target
X = data.drop(columns=["readmitted"])
y = data["readmitted"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (101766, 44)
Target shape: (101766,)


In [18]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=patient_ids)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training data: (81613, 44)
Testing data: (20153, 44)

Training target distribution:
readmitted
0    88.719934
1    11.280066
Name: proportion, dtype: float64

Testing target distribution:
readmitted
0    89.326651
1    10.673349
Name: proportion, dtype: float64


In [19]:
train_patients = set(patient_ids.iloc[train_idx])
test_patients = set(patient_ids.iloc[test_idx])

print("Common patients:", len(train_patients.intersection(test_patients)))

Common patients: 0


In [20]:
# Identify numerical and categorical columns

numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Number of numerical columns:", len(numerical_cols))
print("Numerical columns:")
print(numerical_cols)

print("\nNumber of categorical columns:", len(categorical_cols))
print("Categorical columns:")
print(categorical_cols)

Number of numerical columns: 11
Numerical columns:
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Number of categorical columns: 33
Categorical columns:
['race', 'gender', 'age', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [21]:
print("Missing values in training data:")
print(X_train.isnull().sum().sum())

print("\nMissing values in testing data:")
print(X_test.isnull().sum().sum())

Missing values in training data:
0

Missing values in testing data:
0


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [23]:
numerical_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

In [24]:
categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [26]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("\nOriginal testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

Original training shape: (81613, 44)
Processed training shape: (81613, 2373)

Original testing shape: (20153, 44)
Processed testing shape: (20153, 2373)
